# Freddie Mac Multi-Year Performance Target Validation

## Purpose

This notebook applies the validated 2015 performance-target methodology consistently to the 2015-2019 Freddie Mac annual samples. It follows the same reusable-pipeline, annual-summary, cross-year-comparison, persistence, and read-back pattern used in Notebook 03.

The target definition is fixed before cross-year processing:

- **90+ serious delinquency:** numeric delinquency status 3 or higher, plus `RA`;
- **terminal credit event:** Zero Balance Code `02`, `03`, `09`, or `15`;
- **composite outcome:** either serious delinquency or a terminal credit event;
- **primary horizon:** months 0 through 23 from `FIRST PAYMENT DATE`;
- **eligibility:** history begins no later than month 0 and follow-up is complete through the horizon or a documented termination; and
- **coverage guard:** at least 95% follow-up coverage among loans observable from month 0.

The 12- and 36-month candidates remain available for sensitivity analysis. Overall sample retention is reported separately so left truncation remains visible.

### Documentation basis

- [Freddie Mac Single-Family Loan-Level Dataset General User Guide](https://www.freddiemac.com/fmac-resources/research/pdf/user_guide.pdf)
- [Freddie Mac Non-Standard Dataset Summary Statistics](https://www.freddiemac.com/fmac-resources/research/pdf/nonstandard_dataset_summary_statistics.pdf)

The delinquency and termination meanings come from Freddie Mac documentation. The 24-month horizon and 95% observable-cohort coverage threshold are project methodology choices.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


CURRENT_DIRECTORY = Path.cwd()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.freddie_performance import (
    CREDIT_EVENT_ZERO_BALANCE_CODES,
)
from src.freddie_performance_pipeline import (
    combine_performance_targets,
    run_performance_target_pipeline,
)


print("Notebook directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)

## 1. Multi-Year File Configuration

Define the study window, candidate horizons, selected horizon, coverage threshold, and all file locations once. Every later processing and validation cell uses these shared settings.

Both supported performance filenames are checked because the locally extracted sample files use `sample_perf_YEAR.txt`, while some Freddie Mac releases use `sample_svcg_YEAR.txt`.

In [ ]:
YEARS = [2015, 2016, 2017, 2018, 2019]
HORIZONS = (12, 24, 36)
SELECTED_HORIZON = 24
MINIMUM_OBSERVABLE_COVERAGE_PERCENTAGE = 95.0
EXPECTED_LOANS_PER_YEAR = 50000

RAW_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Freddie_Mac"
)
PROCESSED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Freddie_Mac"
)

PERFORMANCE_FILE_CANDIDATES = {
    year: [
        (
            RAW_DATA_DIRECTORY
            / f"sample_{year}"
            / f"sample_svcg_{year}.txt"
        ),
        (
            RAW_DATA_DIRECTORY
            / f"sample_{year}"
            / f"sample_perf_{year}.txt"
        ),
    ]
    for year in YEARS
}

PERFORMANCE_FILES = {}
ORIGINATION_FILES = {}
file_check_rows = []

for year in YEARS:
    existing_performance_files = [
        path
        for path in PERFORMANCE_FILE_CANDIDATES[year]
        if path.exists()
    ]
    performance_file = (
        existing_performance_files[0]
        if existing_performance_files
        else PERFORMANCE_FILE_CANDIDATES[year][0]
    )
    origination_file = (
        PROCESSED_DATA_DIRECTORY
        / f"origination_{year}_clean.parquet"
    )

    PERFORMANCE_FILES[year] = performance_file
    ORIGINATION_FILES[year] = origination_file

    file_check_rows.extend(
        [
            {
                "year": year,
                "file_type": "raw_performance",
                "exists": performance_file.exists(),
                "path": str(performance_file),
            },
            {
                "year": year,
                "file_type": "cleaned_origination",
                "exists": origination_file.exists(),
                "path": str(origination_file),
            },
        ]
    )

file_check = pd.DataFrame(file_check_rows)
display(file_check)

missing_files = file_check.loc[
    ~file_check["exists"],
    "path",
].tolist()

if missing_files:
    raise FileNotFoundError(
        "Missing multi-year target inputs: "
        f"{missing_files}"
    )

## 2. Confirm the Fixed Target Contract

Display the project decisions before processing any year. These definitions must remain identical across 2015-2019 so differences in event rates are not caused by changing label logic.

Terminal codes supplement the delinquency outcome; they are not silently substituted for it. Codes `01`, `16`, and `96` remain valid termination reasons but do not independently create a positive outcome.

In [ ]:
target_contract = pd.DataFrame(
    [
        {
            "setting": "Study years",
            "value": ", ".join(map(str, YEARS)),
        },
        {
            "setting": "Candidate horizons",
            "value": ", ".join(map(str, HORIZONS)),
        },
        {
            "setting": "Primary horizon",
            "value": f"{SELECTED_HORIZON} months",
        },
        {
            "setting": "Serious delinquency",
            "value": "Numeric status >= 3 or RA",
        },
        {
            "setting": "Terminal credit codes",
            "value": ", ".join(
                sorted(CREDIT_EVENT_ZERO_BALANCE_CODES)
            ),
        },
        {
            "setting": "Coverage denominator",
            "value": "Loans observable from month 0",
        },
        {
            "setting": "Minimum coverage",
            "value": (
                f"{MINIMUM_OBSERVABLE_COVERAGE_PERCENTAGE:.1f}%"
            ),
        },
    ]
)

display(target_contract)

## 3. Run the Reusable Annual Performance Pipeline

Process one annual sample at a time to control memory use. Each run:

1. scans every raw row for the expected 35-field structure;
2. loads only the five fields needed to create the outcome;
3. validates identifiers, periods, codes, duplicates, and origination linkage;
4. builds the first-payment-relative timeline;
5. creates 12-, 24-, and 36-month candidates;
6. applies the observable-cohort coverage guard;
7. saves annual target, horizon, component, cohort, and validation outputs; and
8. reads every analytical output back for verification.

If a year fails a critical validation or the 95% guard, processing stops at that year.

In [ ]:
pipeline_results = {}

for year in YEARS:
    print("=" * 72)
    print(f"Processing performance target for {year}")

    pipeline_result = run_performance_target_pipeline(
        year=year,
        performance_file=PERFORMANCE_FILES[year],
        origination_file=ORIGINATION_FILES[year],
        output_directory=PROCESSED_DATA_DIRECTORY,
        horizons=HORIZONS,
        selected_horizon=SELECTED_HORIZON,
        minimum_observable_coverage_percentage=(
            MINIMUM_OBSERVABLE_COVERAGE_PERCENTAGE
        ),
        expected_loans=EXPECTED_LOANS_PER_YEAR,
    )

    pipeline_results[year] = pipeline_result

    selected = pipeline_result["selected_summary"]

    print(
        "Performance rows:",
        pipeline_result["validation"]["performance_rows"],
    )
    print(
        "Eligible loans:",
        int(selected["eligible_loans"]),
    )
    print(
        "Observable-cohort coverage:",
        f"{selected['observable_cohort_coverage_percentage']:.3f}%",
    )
    print(
        "Composite events:",
        int(selected["composite_credit_events"]),
    )

## 4. Review Annual Structure and Linkage Validation

Combine the lightweight validation reports returned by each annual run. The expected result is one row per year, zero structural errors, 50,000 performance loans matched to 50,000 cleaned origination loans, and no critical key or code violations.

In [ ]:
validation_rows = []

for year in YEARS:
    result = pipeline_results[year]

    validation_rows.append(
        {
            "year": year,
            **result["structure_validation"],
            **result["validation"],
        }
    )

annual_validation = pd.DataFrame(validation_rows)
display(annual_validation)

## 5. Enforce Cross-Year Critical Checks

A successful function call is supplemented by explicit notebook assertions. These checks prevent a partially valid annual result from entering the combined target.

Origination loans without performance are reported separately because they are an eligibility limitation; all other listed fields are critical errors and must be zero.

In [ ]:
critical_zero_columns = [
    "field_count_errors",
    "missing_loan_ids",
    "invalid_periods",
    "duplicate_loan_periods",
    "unexpected_delinquency_statuses",
    "unexpected_zero_balance_codes",
    "unmatched_performance_loans",
]

for column in critical_zero_columns:
    assert annual_validation[column].eq(0).all(), (
        f"Critical validation failures found in {column}."
    )

assert annual_validation[
    "performance_loans"
].eq(EXPECTED_LOANS_PER_YEAR).all(), (
    "A performance file does not contain 50,000 loans."
)

assert annual_validation[
    "origination_loans"
].eq(EXPECTED_LOANS_PER_YEAR).all(), (
    "A cleaned origination file does not contain 50,000 loans."
)

display(
    annual_validation[
        [
            "year",
            *critical_zero_columns,
            "origination_loans_without_performance",
        ]
    ]
)

## 6. Compare 12-, 24-, and 36-Month Horizons Across Years

Combine the annual horizon summaries without pooling away the year. For every year and horizon, retain:

- overall sample retention;
- left-truncated loan count;
- observable-cohort size;
- incomplete follow-up;
- observable-cohort coverage;
- serious-delinquency events; and
- composite credit events and rates.

This makes changes in both target prevalence and target availability visible.

In [ ]:
multi_year_horizon_summary = pd.concat(
    [
        pipeline_results[year]["horizon_summary"]
        for year in YEARS
    ],
    ignore_index=True,
)

rate_columns = [
    "serious_delinquency_rate",
    "composite_credit_event_rate",
]

horizon_summary_display = (
    multi_year_horizon_summary.copy()
)

for column in rate_columns:
    horizon_summary_display[
        f"{column}_percentage"
    ] = horizon_summary_display[column] * 100

display(
    horizon_summary_display.drop(
        columns=rate_columns
    ).round(3)
)

## 7. Verify the 24-Month Coverage Guard for Every Year

Select the primary horizon and verify that each annual observable cohort exceeds the pre-specified 95% follow-up requirement.

Overall retention is not used as the guard because it includes structurally left-truncated loans. It remains reported beside coverage so the excluded share is fully auditable.

In [ ]:
selected_horizon_summary = (
    multi_year_horizon_summary.loc[
        multi_year_horizon_summary[
            "horizon_months"
        ].eq(SELECTED_HORIZON)
    ]
    .sort_values("year")
    .reset_index(drop=True)
)

coverage_columns = [
    "year",
    "total_loans",
    "left_truncated_loans",
    "observable_cohort_loans",
    "incomplete_follow_up_loans",
    "eligible_loans",
    "sample_retention_percentage",
    "observable_cohort_coverage_percentage",
]

display(
    selected_horizon_summary[
        coverage_columns
    ].round(3)
)

assert selected_horizon_summary[
    "observable_cohort_coverage_percentage"
].ge(
    MINIMUM_OBSERVABLE_COVERAGE_PERCENTAGE
).all(), (
    "At least one annual 24-month target failed "
    "the observable-cohort coverage guard."
)

## 8. Compare Composite-Target Components

Display the four mutually exclusive outcome components for each year:

1. 90+ delinquency only;
2. terminal credit event only;
3. both components; and
4. neither component.

The terminal-only row measures exactly how many positives the composite adds beyond observed delinquency status.

In [ ]:
multi_year_component_summary = pd.concat(
    [
        pipeline_results[year]["component_summary"]
        for year in YEARS
    ],
    ignore_index=True,
)

display(
    multi_year_component_summary.round(3)
)

component_reconciliation = (
    multi_year_component_summary.groupby(
        "year",
        as_index=False,
    )
    .agg(component_loans=("loans", "sum"))
    .merge(
        selected_horizon_summary[
            ["year", "eligible_loans"]
        ],
        on="year",
        how="left",
        validate="one_to_one",
    )
)

component_reconciliation["counts_match"] = (
    component_reconciliation["component_loans"]
    == component_reconciliation["eligible_loans"]
)

display(component_reconciliation)

assert component_reconciliation[
    "counts_match"
].all(), "Component counts do not reconcile."

## 9. Visualize Annual Coverage and 24-Month Event Rates

The left panel separates overall sample retention from observable-cohort follow-up coverage. The right panel compares the annual 90+ and composite event rates.

Differences in event rates are descriptive outcome variation. They do not by themselves demonstrate borrower-feature drift, reduced discrimination, or calibration failure.

In [ ]:
figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13, 4.5),
)

year_labels = selected_horizon_summary[
    "year"
].astype(str)

axes[0].plot(
    year_labels,
    selected_horizon_summary[
        "sample_retention_percentage"
    ],
    marker="o",
    label="Overall sample retention",
)
axes[0].plot(
    year_labels,
    selected_horizon_summary[
        "observable_cohort_coverage_percentage"
    ],
    marker="o",
    label="Observable-cohort coverage",
)
axes[0].axhline(
    MINIMUM_OBSERVABLE_COVERAGE_PERCENTAGE,
    color="#E45756",
    linestyle="--",
    label="95% coverage requirement",
)
axes[0].set_title("24-month target availability")
axes[0].set_xlabel("Origination year")
axes[0].set_ylabel("Percentage")
axes[0].legend()

axes[1].plot(
    year_labels,
    selected_horizon_summary[
        "serious_delinquency_rate"
    ] * 100,
    marker="o",
    label="90+ delinquency",
)
axes[1].plot(
    year_labels,
    selected_horizon_summary[
        "composite_credit_event_rate"
    ] * 100,
    marker="o",
    label="Composite credit event",
)
axes[1].set_title("24-month eligible-loan event rates")
axes[1].set_xlabel("Origination year")
axes[1].set_ylabel("Event rate (%)")
axes[1].legend()

figure.suptitle(
    "Freddie Mac 2015-2019 Performance Target Comparison"
)
figure.tight_layout()
plt.show()

## 10. Validate Quarterly Outcome Summaries

Combine the 20 quarterly cohorts from `2015Q1` through `2019Q4`. Verify that every annual file contributes four quarters and that quarterly eligible and event counts reconcile to the annual selected-horizon summary.

Quarterly rates will later help determine whether quarterly, semiannual, or annual evaluation windows contain enough positive events for reliable model monitoring.

In [ ]:
multi_year_quarterly_summary = pd.concat(
    [
        pipeline_results[year]["cohort_summary"]
        for year in YEARS
    ],
    ignore_index=True,
)

quarterly_reconciliation = (
    multi_year_quarterly_summary.groupby(
        "year",
        as_index=False,
    )
    .agg(
        quarter_count=(
            "ORIGINATION COHORT",
            "nunique",
        ),
        quarterly_eligible_loans=(
            "eligible_loans",
            "sum",
        ),
        quarterly_composite_events=(
            "composite_credit_events",
            "sum",
        ),
    )
    .merge(
        selected_horizon_summary[
            [
                "year",
                "eligible_loans",
                "composite_credit_events",
            ]
        ],
        on="year",
        how="left",
        validate="one_to_one",
    )
)

quarterly_reconciliation["eligible_counts_match"] = (
    quarterly_reconciliation[
        "quarterly_eligible_loans"
    ]
    == quarterly_reconciliation["eligible_loans"]
)
quarterly_reconciliation["event_counts_match"] = (
    quarterly_reconciliation[
        "quarterly_composite_events"
    ]
    == quarterly_reconciliation[
        "composite_credit_events"
    ]
)

display(
    multi_year_quarterly_summary.round(3)
)
display(quarterly_reconciliation)

assert quarterly_reconciliation[
    "quarter_count"
].eq(4).all(), "An annual target is missing a quarter."

assert quarterly_reconciliation[
    [
        "eligible_counts_match",
        "event_counts_match",
    ]
].all().all(), "Quarterly counts do not reconcile."

## 11. Visualize the 20 Quarterly Event Rates

Plot the 24-month composite event rate for every origination quarter. This is the outcome series that will later be compared with borrower-feature drift and model-reliability measures.

Low quarterly event counts must be considered before choosing the final monitoring cadence.

In [ ]:
quarterly_plot_data = (
    multi_year_quarterly_summary.sort_values(
        ["year", "ORIGINATION COHORT"]
    )
    .reset_index(drop=True)
)

figure, axis = plt.subplots(figsize=(14, 4.8))

axis.plot(
    quarterly_plot_data["ORIGINATION COHORT"],
    quarterly_plot_data[
        "composite_credit_event_rate_percentage"
    ],
    marker="o",
    color="#2F6690",
)
axis.set_title(
    "24-Month Composite Credit Event Rate by Origination Cohort"
)
axis.set_xlabel("Origination cohort")
axis.set_ylabel("Event rate (%)")
axis.tick_params(axis="x", rotation=45)
axis.grid(axis="y", alpha=0.25)

figure.tight_layout()
plt.show()

## 12. Review Every Annual Read-Back Validation

The annual pipeline reopened each saved candidate Parquet, selected-target Parquet, horizon CSV, cohort CSV, component CSV, and validation JSON.

This table verifies expected candidate rows, unique loan-horizon keys, valid horizons and labels, selected-target uniqueness, summary row counts, count reconciliation, and validation-JSON agreement for each year.

In [ ]:
annual_readback_rows = []

for year in YEARS:
    annual_readback_rows.append(
        {
            "year": year,
            **pipeline_results[year][
                "readback_validation"
            ],
        }
    )

annual_readback_summary = pd.DataFrame(
    annual_readback_rows
)

display(annual_readback_summary)

assert annual_readback_summary[
    "candidate_duplicate_loan_horizons"
].eq(0).all()

assert annual_readback_summary[
    "selected_target_duplicate_loans"
].eq(0).all()

assert annual_readback_summary[
    "validation_json_matches"
].all()

## 13. Create the Combined 2015-2019 Modeling Target

After all annual outputs pass, concatenate the five eligible 24-month target files. Preserve `SOURCE YEAR` and `ORIGINATION COHORT` so later joins, temporal splits, and reliability evaluation remain auditable.

The combined function also saves annual and quarterly summaries and performs a full read-back check.

In [ ]:
combined_results = combine_performance_targets(
    years=YEARS,
    output_directory=PROCESSED_DATA_DIRECTORY,
    selected_horizon=SELECTED_HORIZON,
)

print("Combined outputs:")

for output_name, output_path in (
    combined_results["output_paths"].items()
):
    print(f"{output_name}: {output_path}")

## 14. Validate the Combined Target and Final Outcome Counts

Confirm that the combined file contains all five years, only the selected 24-month horizon, no duplicate year-loan keys, five annual summary rows, and 20 quarterly summary rows.

The annual table is the authoritative cross-year target summary for the next modeling-data join.

In [ ]:
combined_target = combined_results[
    "combined_target"
]
combined_annual_summary = combined_results[
    "annual_summary"
]
combined_quarterly_summary = combined_results[
    "quarterly_summary"
]
combined_readback = combined_results[
    "readback_validation"
]

combined_outcome_summary = (
    combined_target.groupby(
        "SOURCE YEAR",
        as_index=False,
    )
    .agg(
        eligible_loans=("LOAN IDENTIFIER", "size"),
        serious_delinquency_events=(
            "TARGET_90_PLUS",
            "sum",
        ),
        composite_credit_events=(
            "TARGET_COMPOSITE_CREDIT_EVENT",
            "sum",
        ),
        composite_event_rate=(
            "TARGET_COMPOSITE_CREDIT_EVENT",
            "mean",
        ),
    )
)

combined_outcome_summary[
    "composite_event_rate_percentage"
] = (
    combined_outcome_summary[
        "composite_event_rate"
    ] * 100
)

display(
    combined_outcome_summary.drop(
        columns="composite_event_rate"
    ).round(3)
)
display(pd.DataFrame([combined_readback]))

assert combined_readback[
    "duplicate_year_loan_rows"
] == 0
assert combined_readback["saved_years"] == YEARS
assert combined_readback[
    "saved_horizons"
] == [SELECTED_HORIZON]
assert combined_readback[
    "annual_summary_rows"
] == len(YEARS)
assert combined_readback[
    "quarterly_summary_rows"
] == len(YEARS) * 4
assert combined_readback[
    "validation_json_matches"
]

## 15. Conclusion and Next Stage

This notebook completes the reusable 2015-2019 performance-target milestone when all cells pass.

### Completion criteria

- Every annual performance row conforms to the expected 35-field structure.
- All 250,000 origination loans are checked against their monthly performance histories.
- Critical identifier, period, duplicate, code, and linkage checks report zero failures.
- Every annual 24-month target passes the 95% observable-cohort coverage guard.
- Component counts and quarterly counts reconcile to their annual totals.
- Every annual and combined output passes read-back validation.
- The combined target preserves source year, origination quarter, event timing, and both target definitions.

### Interpretation boundary

Annual and quarterly outcome-rate changes are descriptive evidence. Feature drift, model discrimination, calibration, and degradation are separate quantities that must be measured in later stages.

### Next stage

Join the combined 24-month target to the cleaned 2015-2019 origination datasets using `LOAN IDENTIFIER`. Validate one-to-one linkage, exclude unavailable or non-comparable model features according to the documented cleaning rules, and create time-ordered development and out-of-time evaluation datasets before training Logistic Regression and XGBoost.